In [1]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
import torchvision.transforms as transforms

In [2]:
from torchvision import datasets
from torch.utils.data import DataLoader

In [30]:
transform=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    

])

# Load from folder

In [14]:
trainset=datasets.ImageFolder(root=r"C:\Users\admin\Downloads\archive (11)\cats_and_dogs_filtered\train",transform=transform)
testset=datasets.ImageFolder(root=r"C:\Users\admin\Downloads\archive (11)\cats_and_dogs_filtered\validation",transform=transform)

In [15]:
train_loader=DataLoader(trainset,batch_size=32,shuffle=True)
test_loader=DataLoader(testset,batch_size=32)

In [16]:
print(trainset.classes)

['cats', 'dogs']


In [17]:
print(testset.classes)

['cats', 'dogs']


In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv=nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.full_layer=nn.Sequential(
                nn.Linear(64*6*6,128),
                nn.ReLU(),
                nn.Linear(128,2)
            )
    def forward(self,x):
        x=self.conv(x)
        x=x.view(x.size(0),-1)
        x=self.full_layer(x)
        return x
model=CNN()
creteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())



In [19]:
for epoc in range(15):
    running_loss=0.0
    for img,level in train_loader:
        optimizer.zero_grad()
        outputs=model(img)
        loss=creteria(outputs,level)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    print(f"Epoch {epoc+1}, Loss:{running_loss}")

Epoch 1, Loss:43.78341543674469
Epoch 2, Loss:43.237952053546906
Epoch 3, Loss:42.4206748008728
Epoch 4, Loss:40.6125009059906
Epoch 5, Loss:38.14487475156784
Epoch 6, Loss:37.13505461812019
Epoch 7, Loss:35.821398079395294
Epoch 8, Loss:37.42307889461517
Epoch 9, Loss:33.496608197689056
Epoch 10, Loss:33.850116580724716
Epoch 11, Loss:32.12359485030174
Epoch 12, Loss:30.860479593276978
Epoch 13, Loss:29.4340238571167
Epoch 14, Loss:30.04921904206276
Epoch 15, Loss:27.9164287596941


In [20]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 71.9


Improvement of the Model

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv=nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.full_layer=nn.Sequential(
                nn.Linear(64*6*6,128),
                nn.ReLU(),
                nn.Dropout(0.5), # not to memeorize 
                nn.Linear(128,2)
            )
    def forward(self,x):
        x=self.conv(x)
        x=x.view(x.size(0),-1)
        x=self.full_layer(x)
        return x
model=CNN()
creteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [45]:
for epoc in range(40):
    running_loss=0.0
    for img,level in train_loader:
        optimizer.zero_grad()
        outputs=model(img)
        loss=creteria(outputs,level)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    print(f"Epoch {epoc+1}, Loss:{running_loss}")

Epoch 1, Loss:15.666206046938896
Epoch 2, Loss:14.914073057472706
Epoch 3, Loss:15.123282879590988
Epoch 4, Loss:16.38940154016018
Epoch 5, Loss:14.232656352221966
Epoch 6, Loss:13.410171739757061
Epoch 7, Loss:13.702505454421043
Epoch 8, Loss:12.70959908515215
Epoch 9, Loss:13.077628709375858
Epoch 10, Loss:12.950005995109677
Epoch 11, Loss:11.921858616173267
Epoch 12, Loss:11.945430379360914
Epoch 13, Loss:12.181514292955399
Epoch 14, Loss:11.235132522881031
Epoch 15, Loss:12.196690168231726
Epoch 16, Loss:12.068237863481045
Epoch 17, Loss:10.563082471489906
Epoch 18, Loss:10.282755926251411
Epoch 19, Loss:10.528032939881086
Epoch 20, Loss:9.52471511438489
Epoch 21, Loss:10.381892889738083
Epoch 22, Loss:10.512872377410531
Epoch 23, Loss:11.637397825717926
Epoch 24, Loss:9.588607721030712
Epoch 25, Loss:10.399570543318987
Epoch 26, Loss:7.792973451316357
Epoch 27, Loss:10.173403579741716
Epoch 28, Loss:10.460293911397457
Epoch 29, Loss:7.483544846996665
Epoch 30, Loss:8.4279444329440

In [46]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 73.1


In [50]:
from PIL import Image

img = Image.open(r"C:\Users\admin\Downloads\archive (11)\cats_and_dogs_filtered\validation\dogs\dog.2306.jpg")
img = transform(img).unsqueeze(0)

output = model(img)
_, pred = torch.max(output, 1)

print(trainset.classes[pred.item()])

dogs
